# GARCH on Latest R1 Features Dataset

This notebook builds a GARCH baseline on top of the latest `features_YYYYMMDD_HHMMSS.csv` file in `data/processed/`.

The goal is to compare GARCH against SARIMA across:

- three delay targets: `type_1_only`, `type_2_only`, `type_1_and_2`
- three day-type segments: `all_week`, `workday`, `weekend`

The earlier AR-lag GARCH version could produce explosive-looking forecasts because it used a recursive multi-step mean forecast on the delay **level** series. If the AR coefficients are near a non-stationary boundary, long-horizon level forecasts can drift unrealistically even when training data never showed that magnitude.

This notebook uses a more stable comparison baseline:

- split the data by day type the same way the earlier SARIMA workflow did conceptually
- fit a **constant-mean GARCH(1,1)** model within each segment
- evaluate the holdout mean forecast and the conditional volatility forecast

That makes GARCH act as a volatility-aware baseline instead of forcing it to mimic SARIMA's seasonal mean structure.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from arch import arch_model
except ImportError as exc:
    raise ImportError("This notebook requires the `arch` package in the active Jupyter environment.") from exc

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 100)

In [ ]:
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"


def latest_features_file(processed_dir: Path) -> Path:
    candidates = sorted(
        path
        for path in processed_dir.glob("features_*.csv")
        if re.fullmatch(r"features_\d{8}_\d{6}\.csv", path.name)
    )
    if not candidates:
        raise FileNotFoundError("No timestamped features CSV found in data/processed/.")
    return candidates[-1]


def mae(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

In [ ]:
features_path = latest_features_file(PROCESSED_DIR)
df = pd.read_csv(features_path)

for col in ["planned_arrival_dt", "actual_arrival_dt", "hour_trunc", "service_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

df_model = df[df["direction"].notna()].copy()

print(f"Loaded features file: {features_path.name}")
print(f"Original rows: {len(df):,}")
print(f"Rows with non-missing direction: {len(df_model):,}")
print(f"Date range: {df_model['service_date'].min().date()} -> {df_model['service_date'].max().date()}")
display(df_model["day_type"].value_counts(dropna=False))

## 1. Build Hourly Series by Target and Day Segment

We evaluate three target definitions and three segment definitions:

- `all_week`: all rows together
- `workday`: only rows where `day_type == workday`
- `weekend`: only rows where `day_type == weekend`

The `workday` and `weekend` series follow the same broad logic as the earlier SARIMA day-type split: aggregate within the selected subset, then reindex to a regular hourly grid.

In [ ]:
def build_hourly_delay_series(frame: pd.DataFrame, target_col: str, day_type: str | None = None) -> pd.Series:
    subset = frame.copy()
    if day_type is not None:
        subset = subset[subset["day_type"] == day_type].copy()

    subset = subset.dropna(subset=["hour_trunc", target_col])
    if subset.empty:
        return pd.Series(dtype=float)

    hourly = (
        subset.groupby("hour_trunc")[target_col]
        .mean()
        .sort_index()
    )

    full_index = pd.date_range(hourly.index.min(), hourly.index.max(), freq="H")
    hourly = hourly.reindex(full_index)
    hourly = hourly.interpolate(method="time").ffill().bfill()
    hourly.index.name = "timestamp"
    return hourly.astype(float)


df_model["delay_type_1_2_mean"] = df_model[["delay_type_1", "delay_type_2"]].mean(axis=1)

target_specs = {
    "type_1_only": "delay_type_1",
    "type_2_only": "delay_type_2",
    "type_1_and_2": "delay_type_1_2_mean",
}

target_notes = {
    "type_1_only": "Type 1 = last actual arrival - first planned arrival.",
    "type_2_only": "Type 2 = last actual arrival - last planned arrival.",
    "type_1_and_2": "Type 1&2 = row-wise mean of Type 1 and Type 2.",
}

segment_specs = {
    "all_week": None,
    "workday": "workday",
    "weekend": "weekend",
}

series_map = {}
summary_rows = []
for target_label, target_col in target_specs.items():
    for segment_label, segment_day_type in segment_specs.items():
        series = build_hourly_delay_series(df_model, target_col, day_type=segment_day_type)
        combo_key = (target_label, segment_label)
        series_map[combo_key] = series
        summary_rows.append({
            "target_variant": target_label,
            "segment": segment_label,
            "target_column": target_col,
            "n_hours": len(series),
        })

series_summary = pd.DataFrame(summary_rows)
display(series_summary)

In [ ]:
plot_rows = len(target_specs)
plot_cols = len(segment_specs)
fig, axes = plt.subplots(plot_rows, plot_cols, figsize=(18, 4 * plot_rows), sharex=False)
axes = np.atleast_2d(axes)

for row_idx, target_label in enumerate(target_specs):
    for col_idx, segment_label in enumerate(segment_specs):
        ax = axes[row_idx, col_idx]
        series = series_map[(target_label, segment_label)]
        if series.empty:
            ax.set_visible(False)
            continue
        ax.plot(series.index, series.values, color="#4C72B0")
        ax.set_title(f"{target_label} | {segment_label}\n{target_notes[target_label]}")
        ax.set_ylabel("hourly mean delay")
        ax.tick_params(axis="x", rotation=45)

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.show()

## 2. Train/Test Split

We keep the last 72 hours as a holdout window, matching the SARIMA notebook. Combinations with too few observations are skipped.

In [ ]:
FORECAST_HORIZON = 72
MIN_TRAIN_OBS = 7 * 24


def train_test_split_series(series: pd.Series, horizon: int = 72):
    train = series.iloc[:-horizon].copy()
    test = series.iloc[-horizon:].copy()
    return train, test


train_test_splits = {}
split_summary_rows = []
for combo_key, series in series_map.items():
    target_label, segment_label = combo_key
    if len(series) <= FORECAST_HORIZON + MIN_TRAIN_OBS:
        split_summary_rows.append({
            "target_variant": target_label,
            "segment": segment_label,
            "status": "skipped_too_short",
            "train_size": 0,
            "test_size": 0,
        })
        continue

    train, test = train_test_split_series(series, FORECAST_HORIZON)
    train_test_splits[combo_key] = (train, test)
    split_summary_rows.append({
        "target_variant": target_label,
        "segment": segment_label,
        "status": "ready",
        "train_size": len(train),
        "test_size": len(test),
    })

split_summary = pd.DataFrame(split_summary_rows)
display(split_summary)

## 3. Fit Constant-Mean GARCH Baseline

We use a constant-mean `GARCH(1,1)` specification within each segment.

This is deliberately more conservative than the previous AR-lag version:

- it avoids recursive long-horizon mean explosions
- it lets day-type segmentation absorb part of the mean-structure differences
- it keeps the main GARCH role focused on conditional volatility


In [ ]:
def fit_garch_model(series: pd.Series):
    model = arch_model(
        series.astype(float),
        mean="Constant",
        vol="GARCH",
        p=1,
        q=1,
        dist="normal",
        rescale=True,
    )
    return model.fit(disp="off")


garch_results = {}
for combo_key, (train, _) in train_test_splits.items():
    garch_results[combo_key] = fit_garch_model(train)

for combo_key, result in garch_results.items():
    target_label, segment_label = combo_key
    print(f"\n=== {target_label} | {segment_label} ===")
    print(result.summary())

## 4. Forecast on Holdout

For a constant-mean GARCH model, the mean forecast is stable across the horizon while the volatility forecast evolves through the GARCH recursion.

That is the intended behavior here: use GARCH primarily as a variance model and compare the point-forecast quality against SARIMA without forcing unrealistic exponential mean paths.

In [ ]:
def forecast_and_evaluate_garch(result, test: pd.Series, target_label: str, segment_label: str):
    forecast = result.forecast(horizon=len(test), reindex=False)
    pred_mean = pd.Series(
        forecast.mean.iloc[-1].to_numpy(),
        index=test.index,
        name="forecast",
    )
    pred_vol = pd.Series(
        np.sqrt(forecast.variance.iloc[-1].to_numpy()),
        index=test.index,
        name="forecast_volatility",
    )
    metrics = pd.DataFrame({
        "target_variant": [target_label],
        "segment": [segment_label],
        "mae": [mae(test, pred_mean)],
        "rmse": [rmse(test, pred_mean)],
        "train_mean": [float(result.params["mu"])],
    })
    return pred_mean, pred_vol, metrics


forecast_outputs = {}
for combo_key, (_, test) in train_test_splits.items():
    target_label, segment_label = combo_key
    forecast_outputs[combo_key] = forecast_and_evaluate_garch(
        garch_results[combo_key],
        test,
        target_label,
        segment_label,
    )

metrics_table = pd.concat(
    [metrics for _, _, metrics in forecast_outputs.values()],
    ignore_index=True,
).sort_values(["target_variant", "segment"])

display(metrics_table)

In [ ]:
plot_rows = len(target_specs)
plot_cols = len(segment_specs)
fig, axes = plt.subplots(plot_rows, plot_cols, figsize=(18, 4.5 * plot_rows), sharex=False)
axes = np.atleast_2d(axes)

for row_idx, target_label in enumerate(target_specs):
    for col_idx, segment_label in enumerate(segment_specs):
        ax = axes[row_idx, col_idx]
        combo_key = (target_label, segment_label)
        if combo_key not in forecast_outputs:
            ax.set_visible(False)
            continue
        train, test = train_test_splits[combo_key]
        pred, pred_vol, _ = forecast_outputs[combo_key]
        ax.plot(train.index[-7*24:], train.iloc[-7*24:], label="train tail", color="#4C72B0")
        ax.plot(test.index, test, label="actual test", color="#C44E52")
        ax.plot(pred.index, pred, label="forecast", color="#55A868")
        ax.fill_between(pred.index, pred - 1.96 * pred_vol, pred + 1.96 * pred_vol, color="#55A868", alpha=0.2)
        ax.set_title(f"{target_label} | {segment_label}\n{target_notes[target_label]}")
        ax.legend(fontsize=8)

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.show()

## 5. Conditional Volatility Diagnostics

This section focuses on what GARCH adds beyond the mean forecast: time-varying conditional volatility.

In [ ]:
plot_rows = len(target_specs)
plot_cols = len(segment_specs)
fig, axes = plt.subplots(plot_rows, plot_cols, figsize=(18, 4.5 * plot_rows), sharex=False)
axes = np.atleast_2d(axes)

for row_idx, target_label in enumerate(target_specs):
    for col_idx, segment_label in enumerate(segment_specs):
        ax = axes[row_idx, col_idx]
        combo_key = (target_label, segment_label)
        if combo_key not in garch_results:
            ax.set_visible(False)
            continue
        conditional_vol = pd.Series(garch_results[combo_key].conditional_volatility).dropna()
        ax.plot(conditional_vol.index, conditional_vol.values, color="#DD8452")
        ax.set_title(f"Conditional Volatility - {target_label} | {segment_label}")
        ax.tick_params(axis="x", rotation=45)
        ax.set_ylabel("volatility")

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.show()

## 6. Interpretation Guide

Use this notebook together with the SARIMA notebook:

- compare `MAE` and `RMSE` across the same target definitions and day-type segments
- if SARIMA wins clearly on mean forecast error but GARCH highlights volatility spikes, then SARIMA is the stronger point-forecast model and GARCH is better treated as an uncertainty model
- if `workday` and `weekend` behave differently, that supports keeping separate segment-level models instead of one pooled weekly model
- if the `all_week` model performs materially worse than `workday` and `weekend`, that means day-type mixing is washing out structure that matters for forecasting
